In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import torch
import clip
from PIL import Image
from PIL import ImageOps 

import numpy as np
import pandas as pd
import copy
import os
import cv2

import cv2
import pytesseract
import numpy as np


In [ ]:
# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [ ]:

# Define categories
categories = ["car exterior",
              "car interior",
              "trunk interior",
              "steering wheel",
              "car key",
              "engine bay",
              "wheels",
              "moonroof",
              "paperwork",
              "dealership banner",
              "infotainment screen",
              "navigation",
              "gear selector",
              "tachometer",
              "gauge cluster"
]
categories = sorted(set(categories))
print('\n'.join(categories))

In [ ]:

def classify_image(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    text_inputs = clip.tokenize(categories).to(device)

    # Predict category
    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text_inputs)
        probs = (image_features @ text_features.T).softmax(dim=-1)

    # Get best category
    best_match = categories[probs.argmax().item()]
    return best_match

# Example Usage
# image_path = "test_car_image.jpg"
# category = classify_image(image_path)
# print(f"Classified as: {category}")
#



In [ ]:
def get_predicted_categories(image_path:str, categories:list)->dict:
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    text_inputs = clip.tokenize(categories).to(device)
    
    # Predict category
    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text_inputs)
        probs = (image_features @ text_features.T).softmax(dim=-1)

    probs_w_categories = pd.Series(dict(zip(categories, np.array(probs)[0]))).sort_values(ascending=False)
    return probs_w_categories
    


In [ ]:
path_exterior = '/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-695934089/a37219a938b544648009d4a558e393b5.jpg'
path_steering_wheel ='/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-716406739/17fdd1da23284042abe3cb81c2bd0b05.jpg'


In [ ]:



category = classify_image(path_exterior)
print(f"Classified as: {category}")

category = classify_image(path_steering_wheel)
print(f"Classified as: {category}")

## steering wheel again
print(classify_image('/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-716406739/5461dd8a7ce44bbd9e58fa4c427dcf33.jpg'))

print(classify_image('/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-716406739/5461dd8a7ce44bbd9e58fa4c427dcf33.jpg'))

In [ ]:
path_steering_wheel ='/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-716406739/17fdd1da23284042abe3cb81c2bd0b05.jpg'

# steeringwheel = np.array(Image.open(path_steering_wheel).convert("L"))
# steeringwheel = np.array(Image.open(path_steering_wheel))
steeringwheel = Image.open(path_steering_wheel)
steeringwheel_gray = ImageOps.grayscale(steeringwheel)
plt.figure(figsize=(5,5))
# plt.imshow(steeringwheel )
plt.imshow(steeringwheel_gray , cmap='gray')
plt.show()

In [ ]:
## look at one case in detail
image_path = copy.copy(path_steering_wheel)
image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
text_inputs = clip.tokenize(categories).to(device)

# Predict category
with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text_inputs)
    probs = (image_features @ text_features.T).softmax(dim=-1)

# Get best category
best_match = categories[probs.argmax().item()]

print(len(categories),len(probs))
# print(dict(zip(

In [ ]:
dir(probs)
np.array(probs)[0]

dict(zip(categories, np.array(probs)[0]))

# probs.__dict__

In [ ]:
probs_label = pd.Series(dict(zip(categories, np.array(probs)[0])))
probs_label
pred_top3 = probs_label.sort_values(ascending=False).head(3)
dict(pred_top3)

In [ ]:
get_predicted_categories(path_steering_wheel, categories).head(3)

In [ ]:
c2=["car exterior", "car interior", "steering wheel", "car key", "engine bay", "wheels"]
get_predicted_categories(path_steering_wheel, c2).head(3)

In [ ]:
c3=["car exterior", "car interior", ]
get_predicted_categories(path_steering_wheel, c3).head(3)

In [ ]:
# Create formatted text from dictionary
image=steeringwheel
text = "\n".join([f"{k}: {v:.2f}" for k, v in pred_top3.items()])

# Plot the image
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image)
ax.axis("off")  # Hide axis

# Superimpose transparent text box
ax.text(
    0.98, 0.02, text,  # Position (top-right corner)
    fontsize=12, color="white",
    ha="right", va="bottom",
    bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", boxstyle="round,pad=0.3")
)

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Sample prediction dictionary
predictions = pred_top3

# Load image
image = cv2.imread(path_steering_wheel)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for correct display
# image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to RGB for correct display


# Convert image to writable format
overlay = image.copy()

# Define text box properties
x, y, w, h = 10, 10, 350, 80  # Top-right position
alpha = 0.9  # Transparency level

# Draw semi-transparent rectangle
cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)  # Black box
cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)  # Blend with original image

# Add text on top of the rectangle
y_offset = y + 20
for label, prob in predictions.items():
    text = f"{label}: {int(prob*100):.0f}%" 
    cv2.putText(image, text, (x + 10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    y_offset += 20  # Line spacing

# Display the image
plt.imshow(image)
plt.axis("off")  # Hide axes
plt.show()


In [ ]:
def categorize_and_plot(image_path, categories):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB for correct display

    predicted_categories_all = get_predicted_categories(image_path, categories)
    predictions = dict(predicted_categories_all.head(3))
    
    
    # Convert image to writable format
    overlay = image.copy()
    
    # Define text box properties
    x, y, w, h = 10, 10, 350, 80  # Top-right position
    alpha = 0.9  # Transparency level
    
    # Draw semi-transparent rectangle
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)  # Black box
    cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0, image)  # Blend with original image
    
    # Add text on top of the rectangle
    y_offset = y + 20
    for label, prob in predictions.items():
        text = f"{label}: {int(prob*100):.0f}%" 
        cv2.putText(image, text, (x + 10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        y_offset += 20  # Line spacing
    
    # Display the image
    plt.imshow(image)
    plt.axis("off")  # Hide axes
    plt.show()
    return plt 

In [ ]:
a = categorize_and_plot(path_steering_wheel, categories)
b = categorize_and_plot(path_exterior, categories)

In [ ]:
import glob 

files=sorted(glob.glob('/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-x5/vehicle_id-716406739/*.jpg'))
files


In [ ]:

print('\nUsing longer list of categories')
print(categories)

list(map(
    lambda x: categorize_and_plot(image_path=x,categories=categories),
    files
))
                                  
    

In [ ]:
## What are the problems I see so far?
# steering wheel mislabeled as infotainment
# rearview of X5 mislabeled as "wheels"

# main goal is to identify exterior shots and avoid false positives (falsely labeling something as an interior shot)
# False negatives (failing to label a genuine exterior shot as such) will reduce the size of my data set, but not as harmful 

In [ ]:
# let's see if editing the categories will give a better result
categories 

In [ ]:
c2 = ['car exterior',
 'car interior',
 'car key',
 'dealership banner',
 'engine bay',
 'gauge cluster',
 'gear selector',
 'infotainment/navigation screen',
 'moonroof',
 'paperwork',
 'steering wheel',
 'wheels closeup']



print(c2)

list(map(
    lambda x: categorize_and_plot(image_path=x,categories=c2),
    files
))
    

In [ ]:
c3 = ['car exterior',
 'car interior',
 'car key',
 'dealership banner',
 'engine bay',
 'gauge cluster',
'start-stop button',
 'gear selector',
 'infotainment/navigation screen',
 'moonroof',
 'paperwork',
 'steering wheel',
 'wheels closeup']



print(c3)

list(map(
    lambda x: categorize_and_plot(image_path=x,categories=c2),
    files
))
    

In [ ]:
# let's look at a different make/model Honda Odyssey
folder = '/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-honda/model-odyssey/vehicle_id-732088109/'

files=sorted(glob.glob(folder+'*.jpg'))
files

print(c2)

list(map(
    lambda x: categorize_and_plot(image_path=x,categories=c2),
    files
))
    


In [ ]:
c2

In [ ]:
# dealership banner -> advertisement 
# wireless charging 
# dashboard

## one solution also is to just focus on car exterior >= 70%
# there will be false negatives but minimal false positives

In [ ]:
c4 = ['car exterior',
 'car interior',
 'car key',
 # 'dealership banner',
       'advertisement',
      'warranty',
'dashboard',      
'wireless charger',      
 'engine bay',
 'gauge cluster',
 'gear selector',
 'infotainment/navigation screen',
 'moonroof',
 'paperwork',
 'steering wheel',
 'wheels closeup']


print(c4)
for file in files:
    print(os.path.basename(file))
    categorize_and_plot(image_path=file,categories=c4)

In [ ]:
    
# here's one interesting case: 2e4256fdf3e147ea80197b1e3527c1e7
# due to the dealership banner, CLIP thinks it's an advertisement, but it also has a car exterior (2nd highest prob)

image_path='/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-honda/model-odyssey/vehicle_id-732088109/2e4256fdf3e147ea80197b1e3527c1e7.jpg'

categorize_and_plot(image_path,c4)

predicted_categories_all = get_predicted_categories(image_path, c4)
predictions = dict(predicted_categories_all.head(3))
predictions


In [ ]:
image_path_cropped = image_path.replace('.jpg','-cropped.jpg')
categorize_and_plot(image_path_cropped, c4)


In [ ]:
predicted_categories_all = get_predicted_categories(image_path, c4+['phone number'])
predictions = dict(predicted_categories_all.head(3))
predictions

In [ ]:
predicted_categories_all = get_predicted_categories(image_path, c4+['business information'])
predictions = dict(predicted_categories_all.head(3))
predictions

In [ ]:
file_cropped = '/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-bmw/model-4-series/vehicle_id-735462554/279fbaf57ca9488997e5b1a4189e6596-cropped.jpg'
file_not_cropped = file_cropped.replace('-cropped','')

In [ ]:

categorize_and_plot(file_not_cropped, c4)
categorize_and_plot(file_cropped, c4)


In [ ]:
import pytesseract

In [ ]:
image = cv2.imread(file_not_cropped)

# cv2.imshow("Text Detection", image)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
plt.imshow(image)
plt.axis("off")  # Hide axis
plt.show()

In [ ]:
image = cv2.imread(file_not_cropped)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
plt.imshow(gray, cmap='gray')
plt.axis("off")  # Hide axis
plt.show()

In [ ]:
# image = cv2.imread("car_image.jpg")

# Convert to grayscale
# gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply thresholding to enhance text visibility
thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

# Detect text regions using Tesseract
custom_config = r'--oem 3 --psm 6'  # OCR engine mode and page segmentation mode
data = pytesseract.image_to_data(thresh, config=custom_config, output_type=pytesseract.Output.DICT)

# Draw bounding boxes around detected text
for i in range(len(data['text'])):
    # print(i)
    if int(data['conf'][i]) > 50:  # Confidence threshold
        print(data['text'][i], data['conf'][i])
        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
        cv2.rectangle(gray, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.imshow(gray, cmap='gray')
plt.axis("off")  # Hide axis
plt.show()

In [ ]:
image = cv2.imread(file_not_cropped)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
image_height = image.shape[0]

# Apply thresholding to enhance text visibility
thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
thresh = gray


# Detect text regions using Tesseract
custom_config = r'--oem 3 --psm 3'  # OCR engine mode and page segmentation mode
# custom_config = r'--oem 3 --psm 7'  # OCR engine mode and page segmentation mode
data = pytesseract.image_to_data(thresh, config=custom_config, output_type=pytesseract.Output.DICT)

# Draw bounding boxes around detected text
for i in range(len(data['text'])):
    if int(data['conf'][i]) > 50:  # Confidence threshold
        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
        height_pct = int(100*y/image_height)
        depth_pct = int(100*h/image_height)
        is_banner = all([
            depth_pct <= 20,
            (height_pct <= 10 or height_pct >= 90)
        ])
        
        # if height_pct < 10 or height_pct > 90:
        if is_banner:
            print(data['text'][i], data['conf'][i], f'{height_pct}%')
            cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.imshow(image)
plt.axis("off")  # Hide axis
plt.show()

In [ ]:
plt.imshow(thresh, cmap='gray')
plt.axis("off")  # Hide axis
plt.show()

In [ ]:
del image

In [ ]:
def preprocess_image_for_banner_text(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    return thresh


def banner_text_box(image_path, preprocessing_fn):

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    
    image_height = image.shape[0]
    
    # Apply thresholding to enhance text visibility
    image_processed = preprocessing_fn(image)
    
    # Detect text regions using Tesseract
    custom_config = r'--oem 3 --psm 3'  # OCR engine mode and page segmentation mode
    data = pytesseract.image_to_data(image_processed, config=custom_config, output_type=pytesseract.Output.DICT)
    
    # Draw bounding boxes around detected text
    data['is_banner']=[0] * len(data['text'])
    for i in range(len(data['text'])):
        if int(data['conf'][i]) > 50:  # Confidence threshold
            x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
            height_pct = int(100*y/image_height)
            depth_pct = int(100*h/image_height)
            is_banner = all([
                depth_pct <= 20,
                (height_pct <= 10 or height_pct >= 90)
            ])
            if is_banner:
                data['is_banner'][i] = 1 # int(is_banner)
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # if plot:
    plt.imshow(image)
    plt.axis("off")  # Hide axis
    plt.show()

    return plt,data

In [ ]:
# image = cv2.imread(file_not_cropped)
plt,text_data = banner_text_box(file_not_cropped, preprocess_image_for_banner_text)

In [ ]:
# image = cv2.imread(file_not_cropped)
plt,text_data = banner_text_box(file_not_cropped, preprocess_image_for_banner_text)

In [ ]:
image_path='/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/make-honda/model-odyssey/vehicle_id-732088109/2e4256fdf3e147ea80197b1e3527c1e7.jpg'
plt,text_data = banner_text_box(image_path, preprocess_image_for_banner_text)

In [ ]:
plt.imshow(gray, cmap='gray')

In [ ]:
## let's try EAST

import cv2
import pytesseract
import numpy as np

# Load the image
image = cv2.imread(image_path)
# resize 32
height, width = image.shape[:2]
new_width = (width // 32) * 32
new_height = (height // 32) * 32
image = cv2.resize(image, (new_width, new_height))



# Load the pre-trained EAST text detector
east_model = "/Users/levgolod/Projects/ml_tools/frozen_east_text_detection.pb"
net = cv2.dnn.readNet(east_model)

# Prepare input for the network
height, width = image.shape[0], image.shape[1]
blob = cv2.dnn.blobFromImage(image, 1.0, (width, height),
                             (123.68, 116.78, 103.94), swapRB=True, crop=False)
net.setInput(blob)

# Get text detection output
scores, geometry = net.forward(["feature_fusion/Conv_7/Sigmoid", "feature_fusion/concat_3"])

In [ ]:
# Process detected text boxes
def decode_predictions(scores, geometry, conf_threshold=0.5):
    detections = []
    for y in range(scores.shape[2]):
        for x in range(scores.shape[3]):
            if scores[0, 0, y, x] > conf_threshold:
                offset_x, offset_y = x * 4, y * 4
                angle = geometry[0, 4, y, x]
                cos, sin = np.cos(angle), np.sin(angle)
                h, w = geometry[0, 0, y, x], geometry[0, 1, y, x]
                x1, y1 = int(offset_x - w / 2), int(offset_y - h / 2)
                x2, y2 = int(offset_x + w / 2), int(offset_y + h / 2)
                detections.append((x1, y1, x2, y2))
    return detections

# Get bounding boxes
boxes = decode_predictions(scores, geometry)

# Draw and extract text regions
for (x1, y1, x2, y2) in boxes:
    # Ensure coordinates are within bounds
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(width, x2)
    y2 = min(height, y2)
    roi = image[y1:y2, x1:x2]
    text = pytesseract.image_to_string(roi, config="--oem 3 --psm 7")
    print("Detected Text:", text)

# Show results
for (x1, y1, x2, y2) in boxes:
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

cv2.imshow("Detected Text", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
## not too happy w EAST; let's try paddle

from paddleocr import PaddleOCR
import cv2

# Initialize PaddleOCR
ocr = PaddleOCR(lang="en")

# Load the image
# image_path = "car_banner.jpg"
result = ocr.ocr(image_path, cls=True)

# Show results
image = cv2.imread(image_path)
for line in result[0]:
    text, confidence = line[1]
    print(f"Detected: {text} (Confidence: {confidence:.2f})")


In [ ]:
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    
    image_height = image.shape[0]
    
    # Apply thresholding to enhance text visibility
    image_processed = preprocessing_fn(image)
    
    # Detect text regions using Tesseract
    custom_config = r'--oem 3 --psm 3'  # OCR engine mode and page segmentation mode
    data = pytesseract.image_to_data(image_processed, config=custom_config, output_type=pytesseract.Output.DICT)
    
    # Draw bounding boxes around detected text
    data['is_banner']=[0] * len(data['text'])
    for i in range(len(data['text'])):
        if int(data['conf'][i]) > 50:  # Confidence threshold
            x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
            height_pct = int(100*y/image_height)
            depth_pct = int(100*h/image_height)
            is_banner = all([
                depth_pct <= 20,
                (height_pct <= 10 or height_pct >= 90)
            ])
            if is_banner:
                data['is_banner'][i] = 1 # int(is_banner)
                cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    plt.imshow(image)
    plt.axis("off")  # Hide axis
    plt.show()
    return plt,data

In [ ]:
# largest folders
# find '/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/' 
!du -h -d 5 '/Users/levgolod/Projects/car_classifier/data/autotrader/vehicle_images/'  | sort -hr | grep 'vehicle_id' | head 
